In [138]:
import random
import h5py
import numpy as np
import pandas as pd

In [139]:
# Spheres
with open(
    "/home/dbruwel/main/honours/pachner_graph_triangulations/data/input_data/dehydration/raw/spheres_10.txt",
    "r",
) as f:
    sphere_data = f.readlines()
    sphere_data = [t.strip() for t in sphere_data]
    random.shuffle(sphere_data)

sphere_data = pd.DataFrame(sphere_data, columns=['isos'])
sphere_data['is_manifold'] = 1
sphere_data['triangulation_type'] = 's3'
sphere_data['is_test'] = (sphere_data.index < 1000).astype(int)

In [147]:
# Random gluings
with open(
    "/home/dbruwel/main/honours/pachner_graph_triangulations/data/input_data/dehydration/raw/random_10.txt",
    "r",
) as f:
    random_data = f.readlines()
    random_data = [t.strip() for t in random_data]
    random.shuffle(random_data)

random_data = pd.DataFrame(random_data, columns=['isos'])
random_data['is_manifold'] = 0
random_data['triangulation_type'] = 'random'
random_data['is_test'] = (random_data.index < 1000).astype(int)

In [148]:
# Near misses
with open(
    "/home/dbruwel/main/honours/pachner_graph_triangulations/data/input_data/dehydration/raw/near_miss_10.txt",
    "r",
) as f:
    near_miss_data = f.readlines()
    near_miss_data = [t.strip() for t in near_miss_data]
    random.shuffle(near_miss_data)

near_miss_data = pd.DataFrame(near_miss_data, columns=['isos'])
near_miss_data['is_manifold'] = 0
near_miss_data['triangulation_type'] = 'near_miss'
near_miss_data['is_test'] = (near_miss_data.index < 1000).astype(int)

In [141]:
# Random manifolds
with open(
    "/home/dbruwel/main/honours/pachner_graph_triangulations/data/input_data/dehydration/raw/random_manifolds_10.txt",
    "r",
) as f:
    random_manifold_data = f.readlines()
    random_manifold_data = [t.strip() for t in random_manifold_data]
    random.shuffle(random_manifold_data)

random_manifold_data = pd.DataFrame(random_manifold_data, columns=['isos'])
random_manifold_data['is_manifold'] = 1
random_manifold_data['triangulation_type'] = 'random_manifold'
random_manifold_data['is_test'] = (random_manifold_data.index < 1000).astype(int)

In [142]:
# Random other test manifolds
from pathlib import Path

labled_manifold_data = {}

path = Path("/home/dbruwel/main/honours/pachner_graph_triangulations/data/input_data/dehydration/raw/manifold_difference")
for f in path.iterdir():
    if f.is_file() and f.suffix == ".txt":
        if f.stem == 's3':
            continue
        with open(f, "r") as file:
            manifold_data = file.readlines()
            manifold_data = [t.strip() for t in manifold_data]
            random.shuffle(manifold_data)
            labled_manifold_data[f.stem] = manifold_data

labled_manifold_data = pd.DataFrame(labled_manifold_data).melt(var_name='triangulation_type', value_name='isos')
labled_manifold_data['is_manifold'] = 1
labled_manifold_data['is_test'] = 1

In [150]:
all_data = pd.concat(
    [sphere_data, random_manifold_data, random_data, near_miss_data, labled_manifold_data],
    ignore_index=True,
)
all_data = all_data.sort_values(by="is_test", ascending=False)

isos = all_data["isos"].to_numpy()
triangulation_type = all_data["triangulation_type"].to_numpy()
is_manifold = all_data["is_manifold"].to_numpy()
is_test = all_data["is_test"].to_numpy()

In [156]:
with h5py.File('classification_10.hdf5', 'w') as hdf_file:
    string_dtype = h5py.string_dtype(encoding='utf-8')

    hdf_file.create_dataset(
        name='isos',
        data=isos,
        dtype=string_dtype
    )

    hdf_file.create_dataset(
        name='triangulation_type',
        data=triangulation_type,
        dtype=string_dtype
    )

    hdf_file.create_dataset(
        name='is_manifold',
        data=is_manifold,
        dtype=np.int32
    )

    hdf_file.create_dataset(
        name='is_test',
        data=is_test,
        dtype=np.int32
    )
